In [1]:
import pandas as pd
import numpy as np

from mlxtend.frequent_patterns import apriori, association_rules

In [2]:
df = pd.read_csv('weatherHistory.csv')

print("Kích thước dữ liệu:", df.shape)
df.head()

Kích thước dữ liệu: (96453, 12)


,Formatted Date,Summary,Precip Type,Temperature (C),Apparent Temperature (C),Humidity,Wind Speed (km/h),Wind Bearing (degrees),Visibility (km),Loud Cover,Pressure (millibars),Daily Summary
0,2006-04-01 00:00:00.000 +0200,Partly Cloudy,rain,9.472222,7.388889,0.89,14.1197,251.0,15.8263,0.0,1015.13,Partly cloudy throughout the day.
1,2006-04-01 01:00:00.000 +0200,Partly Cloudy,rain,9.355556,7.227778,0.86,14.2646,259.0,15.8263,0.0,1015.63,Partly cloudy throughout the day.
2,2006-04-01 02:00:00.000 +0200,Mostly Cloudy,rain,9.377778,9.377778,0.89,3.9284,204.0,14.9569,0.0,1015.94,Partly cloudy throughout the day.
3,2006-04-01 03:00:00.000 +0200,Partly Cloudy,rain,8.288889,5.944444,0.83,14.1036,269.0,15.8263,0.0,1016.41,Partly cloudy throughout the day.
4,2006-04-01 04:00:00.000 +0200,Mostly Cloudy,rain,8.755556,6.977778,0.83,11.0446,259.0,15.8263,0.0,1016.51,Partly cloudy throughout the day.


In [3]:
data = df[['Temperature (C)', 'Humidity', 'Wind Speed (km/h)', 'Summary']].copy()

In [4]:
# Số → median
data['Temperature (C)'].fillna(data['Temperature (C)'].median(), inplace=True)
data['Humidity'].fillna(data['Humidity'].median(), inplace=True)
data['Wind Speed (km/h)'].fillna(data['Wind Speed (km/h)'].median(), inplace=True)

# Chuỗi → mode
data['Summary'].fillna(data['Summary'].mode()[0], inplace=True)

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_12532\4011346073.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  data['Temperature (C)'].fillna(data['Temperature (C)'].median(), inplace=True)
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_12532\4011346073.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values alwa

In [5]:
# Nhiệt độ
data['Temp_Level'] = pd.cut(
    data['Temperature (C)'],
    bins=[-50, 10, 25, 50],
    labels=['Low', 'Medium', 'High']
)

# Độ ẩm
data['Humidity_Level'] = pd.cut(
    data['Humidity'],
    bins=[0, 0.5, 1],
    labels=['Low', 'High']
)

# Gió
data['Wind_Level'] = pd.cut(
    data['Wind Speed (km/h)'],
    bins=[0, 10, 25, 100],
    labels=['Weak', 'Medium', 'Strong']
)

In [6]:
transaction_data = pd.get_dummies(
    data[['Temp_Level', 'Humidity_Level', 'Wind_Level', 'Summary']]
)

print("Dữ liệu sau khi encode:")
transaction_data.head()

Dữ liệu sau khi encode:


,Temp_Level_Low,Temp_Level_Medium,Temp_Level_High,Humidity_Level_Low,Humidity_Level_High,Wind_Level_Weak,Wind_Level_Medium,Wind_Level_Strong,Summary_Breezy,Summary_Breezy and Dry,...,Summary_Mostly Cloudy,Summary_Overcast,Summary_Partly Cloudy,Summary_Rain,Summary_Windy,Summary_Windy and Dry,Summary_Windy and Foggy,Summary_Windy and Mostly Cloudy,Summary_Windy and Overcast,Summary_Windy and Partly Cloudy
0,True,False,False,False,True,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
1,True,False,False,False,True,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
2,True,False,False,False,True,True,False,False,False,False,...,True,False,False,False,False,False,False,False,False,False
3,True,False,False,False,True,False,True,False,False,False,...,False,False,True,False,False,False,False,False,False,False
4,True,False,False,False,True,False,True,False,False,False,...,True,False,False,False,False,False,False,False,False,False


In [7]:
frequent_items = apriori(
    transaction_data,
    min_support=0.05,
    use_colnames=True
)

print("Frequent Itemsets:")
frequent_items.sort_values(by='support', ascending=False).head(10)

Frequent Itemsets:


,support,itemsets
4,0.838543,(Humidity_Level_High)
5,0.488103,(Wind_Level_Weak)
1,0.463169,(Temp_Level_Medium)
6,0.454356,(Wind_Level_Medium)
0,0.443294,(Temp_Level_Low)
12,0.433973,"(Temp_Level_Low, Humidity_Level_High)"
32,0.422786,"(Wind_Level_Weak, Humidity_Level_High)"
20,0.385224,"(Temp_Level_Medium, Humidity_Level_High)"
33,0.368293,"(Wind_Level_Medium, Humidity_Level_High)"
11,0.329000,(Summary_Partly Cloudy)


In [8]:
rules = association_rules(
    frequent_items,
    metric='confidence',
    min_threshold=0.6
)

print("Association Rules:")
rules.head()

Association Rules:


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,(Temp_Level_Low),(Humidity_Level_High),0.443294,0.838543,0.433973,0.978974,1.167470,1.0,0.062252,7.678990,0.257671,0.511843,0.869775,0.748253
1,(Summary_Foggy),(Temp_Level_Low),0.074109,0.443294,0.068728,0.927392,2.092050,1.0,0.035876,7.667316,0.563781,0.153180,0.869576,0.541216
2,(Summary_Overcast),(Temp_Level_Low),0.172073,0.443294,0.110209,0.640477,1.444815,1.0,0.033930,1.548459,0.371856,0.218168,0.354197,0.444546
3,(Temp_Level_Medium),(Humidity_Level_High),0.463169,0.838543,0.385224,0.831714,0.991856,1.0,-0.003163,0.959421,-0.015064,0.420326,-0.042296,0.645555
4,(Temp_Level_High),(Humidity_Level_Low),0.093538,0.161229,0.074192,0.793172,4.919545,1.0,0.059111,4.055409,0.878944,0.410863,0.753416,0.626668


In [9]:
rules = rules.sort_values(by='lift', ascending=False)

rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10)

,antecedents,consequents,support,confidence,lift
4,(Temp_Level_High),(Humidity_Level_Low),0.074192,0.793172,4.919545
19,(Summary_Foggy),"(Temp_Level_Low, Humidity_Level_High)",0.068645,0.926273,2.134402
18,"(Summary_Foggy, Humidity_Level_High)",(Temp_Level_Low),0.068645,0.927441,2.092159
1,(Summary_Foggy),(Temp_Level_Low),0.068728,0.927392,2.092050
5,(Temp_Level_High),(Summary_Partly Cloudy),0.059677,0.637996,1.939200
35,(Summary_Foggy),"(Wind_Level_Weak, Humidity_Level_High)",0.051932,0.700755,1.657470
44,"(Wind_Level_Medium, Summary_Overcast, Humidity...",(Temp_Level_Low),0.061180,0.660363,1.489673
45,"(Wind_Level_Medium, Summary_Overcast)","(Temp_Level_Low, Humidity_Level_High)",0.061180,0.641553,1.478323
22,"(Summary_Overcast, Humidity_Level_High)",(Temp_Level_Low),0.109618,0.653421,1.474013
23,(Summary_Overcast),"(Temp_Level_Low, Humidity_Level_High)",0.109618,0.637043,1.467932


In [10]:
print("\n=== CÁC LUẬT KẾT HỢP ===\n")

for i, row in rules.head(10).iterrows():
    print(f"Luật: {set(row['antecedents'])} → {set(row['consequents'])}")
    print(f"Support: {row['support']:.2f}")
    print(f"Confidence: {row['confidence']:.2f}")
    print(f"Lift: {row['lift']:.2f}")
    print("------")


=== CÁC LUẬT KẾT HỢP ===

Luật: {'Temp_Level_High'} → {'Humidity_Level_Low'}
Support: 0.07
Confidence: 0.79
Lift: 4.92
------
Luật: {'Summary_Foggy'} → {'Temp_Level_Low', 'Humidity_Level_High'}
Support: 0.07
Confidence: 0.93
Lift: 2.13
------
Luật: {'Summary_Foggy', 'Humidity_Level_High'} → {'Temp_Level_Low'}
Support: 0.07
Confidence: 0.93
Lift: 2.09
------
Luật: {'Summary_Foggy'} → {'Temp_Level_Low'}
Support: 0.07
Confidence: 0.93
Lift: 2.09
------
Luật: {'Temp_Level_High'} → {'Summary_Partly Cloudy'}
Support: 0.06
Confidence: 0.64
Lift: 1.94
------
Luật: {'Summary_Foggy'} → {'Wind_Level_Weak', 'Humidity_Level_High'}
Support: 0.05
Confidence: 0.70
Lift: 1.66
------
Luật: {'Wind_Level_Medium', 'Summary_Overcast', 'Humidity_Level_High'} → {'Temp_Level_Low'}
Support: 0.06
Confidence: 0.66
Lift: 1.49
------
Luật: {'Wind_Level_Medium', 'Summary_Overcast'} → {'Temp_Level_Low', 'Humidity_Level_High'}
Support: 0.06
Confidence: 0.64
Lift: 1.48
------
Luật: {'Summary_Overcast', 'Humidity_Level